In [7]:
import joblib
import pandas as pd

MODEL_PATH = "../invoice_flagging/models/predict_flag_invoice.pkl"
SCALER_PATH = "../invoice_flagging/models/scaler.pkl"
EXPECTED_FEATURES = [
    "invoice_quantity",
    "invoice_dollars",
    "invoice_freight",
    "total_brands",
    "total_item_quantity",
    "days_po_to_invoice",
    "total_item_dollars",
]


def load_model(model_path: str = MODEL_PATH):
    """Load the trained invoice flag classifier."""
    with open(model_path, "rb") as f:
        model = joblib.load(f)
    return model


def load_scaler(scaler_path: str = SCALER_PATH):
    """Load the scaler used during training (if available)."""
    with open(scaler_path, "rb") as f:
        scaler = joblib.load(f)
    return scaler


def _coerce_input(input_data) -> pd.DataFrame:
    """Normalize input to the exact training feature schema."""
    input_df = pd.DataFrame(input_data).copy()

    for col in EXPECTED_FEATURES:
        if col not in input_df.columns:
            input_df[col] = 0

    input_df = input_df[EXPECTED_FEATURES]
    return input_df


def predict_invoice_flag(input_data, use_scaler: bool = True):
    """Predict invoice flags for new vendor invoices.

    Parameters
    ----------
    input_data : dict | list[dict] | pd.DataFrame
        Input rows containing invoice-level features.
    use_scaler : bool
        Apply the saved scaler before prediction.

    Returns
    -------
    pd.DataFrame
        Input data with a new `Predicted_Flag` column.
    """
    model = load_model()
    input_df = _coerce_input(input_data)

    if use_scaler:
        scaler = load_scaler()
        X_pred = scaler.transform(input_df)
    else:
        X_pred = input_df

    result_df = input_df.copy()
    result_df["Predicted_Flag"] = model.predict(X_pred).round()
    return result_df


if __name__ == "__main__":
    sample_input = {
        "invoice_quantity": [20, 100],
        "invoice_dollars": [1000, 5000],
        "invoice_freight": [45, 210],
        "total_brands": [3, 8],
        "total_item_quantity": [120, 700],
        "days_po_to_invoice": [7, 24],
        "total_item_dollars": [955, 4790],
    }
    predictions = predict_invoice_flag(sample_input)
    print(predictions)

   invoice_quantity  invoice_dollars  invoice_freight  total_brands  \
0                20             1000               45             3   
1               100             5000              210             8   

   total_item_quantity  days_po_to_invoice  total_item_dollars  Predicted_Flag  
0                  120                   7                 955               0  
1                  700                  24                4790               1  
